In [ ]:
import torch
print('PyTorch  :', torch.__version__)
print('Device   :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('CUDA     :', torch.cuda.is_available())

# 1. Config & Path

In [ ]:
import os
import zipfile
from huggingface_hub import hf_hub_download
from PIL import Image
import matplotlib.pyplot as plt

repo_id = "mohanty/PlantVillage"

# 1. Download the zipped data archive
print("Downloading data.zip... This may take a moment.")
zip_path = hf_hub_download(repo_id=repo_id, filename="data.zip", repo_type="dataset")

# 2. Extract the zip file
extract_path = "/content/plantdisease_data"
if not os.path.exists(extract_path):
    print("Extracting data.zip...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Extraction complete!")
else:
    print("Data already extracted.")

# 3. Load and plot the first training image
# relative_path = train_set[0]['text']
# full_image_path = os.path.join(extract_path, relative_path)

# try:
#     img = Image.open(full_image_path)
#     plt.figure(figsize=(6, 6))
#     plt.imshow(img)
#     plt.title(relative_path.split('/')[-2])
#     plt.axis('off')
#     plt.show()
# except Exception as e:
#     print(f"Could not load the image: {e}\nExpected path: {full_image_path}")

In [ ]:
import os
import json
import subprocess
import zipfile

# IMAGES_FOLDERS_PATH = '/content/plantdisease_data/PlantVillage'
# OUTPUT_FOLDER       = '/content/plantdisease_split'

# def setup_and_download(dataset_name, username, api_token, output_dir='dataset'):
#     if os.path.exists(output_dir) and os.path.isdir(output_dir) and len(os.listdir(output_dir)) > 0:
#         print(f"Dataset already present in '{output_dir}'. Skipping download.")
#         return

#     kaggle_dir = os.path.expanduser('~/.kaggle')
#     os.makedirs(kaggle_dir, exist_ok=True)

#     dest_json = os.path.join(kaggle_dir, 'kaggle.json')
#     with open(dest_json, 'w') as f:
#         json.dump({"username": username, "key": api_token}, f)
#     os.chmod(dest_json, 0o600)

#     print("kaggle credentials configured.")

#     print(f"Downloading {dataset_name}...")
#     subprocess.run(['kaggle', 'datasets', 'download', '-d', dataset_name], check=True)

#     # 3. Identify and extract the zip file
#     zip_filename = f"{dataset_name.split('/')[-1]}.zip"
#     if os.path.exists(zip_filename):
#         print(f"Extracting to '{output_dir}' directory...")
#         os.makedirs(output_dir, exist_ok=True)
#         with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
#             zip_ref.extractall(output_dir)

#         os.remove(zip_filename)
#         print("Download and extraction complete.")
#     else:
#         print("Error: Zip file not found.")

In [ ]:
# setup_and_download(
#     dataset_name='emmarex/plantdisease',
#     username='vinunain',
#     api_token='KGAT_cc1b895cf27e0d44db2639d4acb8abc3',
#     output_dir='plantdisease_data'
# )

# paths
IMAGES_FOLDERS_PATH = '/content/plantdisease_data/raw/color'
OUTPUT_FOLDER       = '/content/plantdisease_split'

# image dimensions
WIDTH    = 224
HEIGHT   = 224
CHANNELS = 3

LBP_RADIUS   = 3
LBP_N_POINTS = 8 * LBP_RADIUS
LBP_METHOD   = 'uniform'


# 2. EDA

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


data = []
for class_name in os.listdir(IMAGES_FOLDERS_PATH):
    class_path = os.path.join(IMAGES_FOLDERS_PATH, class_name)
    if os.path.isdir(class_path):
        count = len(os.listdir(class_path))
        data.append({'Class': class_name, 'Count': count})

df_dist = pd.DataFrame(data).sort_values(by='Count', ascending=False)

print("Image distribution across classes:")
print(df_dist.to_string(index=False))

plt.figure(figsize=(12, 6))
plt.barh(df_dist['Class'], df_dist['Count'], color='skyblue')
plt.xlabel('Number of Images')
plt.ylabel('Class Name')
plt.title('Class Distribution (Number of images per class)')
plt.gca().invert_yaxis()
plt.grid(axis='x', linestyle='--')
plt.show()

## Visualize images

In [ ]:
from PIL import Image
import matplotlib.image as mpimg

def VisualizeClasses(folders_path):
    # Get all class folders and sort them
    ClassesNames = os.listdir(folders_path)
    ClassesNames.sort(key=lambda x: (x.split('__')[0], x))

    # Display the total number of classes
    NUM_CLASSES = len(ClassesNames)
    print(f"Classes Number: {NUM_CLASSES}")

    cols = 5
    rows = (NUM_CLASSES + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 3))
    axes = axes.flatten()

    # Show one sample image from each class
    for i, cat in enumerate(ClassesNames):
        ClassFolderPath = os.path.join(folders_path,cat)
        ImagesName = os.listdir(ClassFolderPath)
        img_path = os.path.join(ClassFolderPath,ImagesName[0])
        with Image.open(img_path) as img:
            width, height = img.size
        image = mpimg.imread(img_path)
        axes[i].imshow(image)
        axes[i].set_title(f"{cat}\n{width}x{height}", fontsize=10)
        axes[i].axis('off')

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

VisualizeClasses(IMAGES_FOLDERS_PATH)

In [ ]:
from numpy import random

# Pick a random class folder
ClassesNames = os.listdir(IMAGES_FOLDERS_PATH)
random_class = random.choice(ClassesNames)
class_path = os.path.join(IMAGES_FOLDERS_PATH, random_class)

# Pick a random image from that class
images = os.listdir(class_path)
random_image = random.choice(images)
img_path = os.path.join(class_path, random_image)

# Open and resize the selected image to 224x224
with Image.open(img_path) as img:
    resized_img = img.resize((WIDTH, HEIGHT))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(mpimg.imread(img_path))
axes[0].set_title("Original")
axes[0].axis('off')

axes[1].imshow(resized_img)
axes[1].set_title(f"Resized ({WIDTH}x{HEIGHT})")
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import cv2

# plotting healty vs diseased plant

def plot_healthy_vs_diseased(plant_name):
    all_classes = [d for d in os.listdir(IMAGES_FOLDERS_PATH) if d.startswith(plant_name)]

    healthy_class = [c for c in all_classes if 'healthy' in c.lower()]
    diseased_classes = [c for c in all_classes if 'healthy' not in c.lower()]

    if not healthy_class or not diseased_classes:
        print(f"Couldn't find both healthy and diseased for {plant_name}")
        return

    target_diseased = random.choice(diseased_classes)
    target_healthy = healthy_class[0]

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    for i, class_name in enumerate([target_healthy, target_diseased]):
        class_path = os.path.join(IMAGES_FOLDERS_PATH, class_name)
        img_name = random.choice(os.listdir(class_path))
        img_path = os.path.join(class_path, img_name)

        img_bgr = cv2.imread(img_path)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        axes[0, i].imshow(img_rgb)
        axes[0, i].set_title(f"Class: {class_name}")
        axes[0, i].axis('off')

        colors = ('red', 'green', 'blue')
        for j, col in enumerate(colors):
            hist = cv2.calcHist([img_rgb], [j], None, [256], [0, 256])
            axes[1, i].plot(hist, color=col, label=col)
            axes[1, i].set_xlim([0, 256])

        axes[1, i].set_title(f"Color Histogram ({'Healthy' if i==0 else 'Diseased'})")
        axes[1, i].legend()

    plt.tight_layout()
    plt.show()

plot_healthy_vs_diseased('Potato')
plot_healthy_vs_diseased('Pepper')
plot_healthy_vs_diseased('Tomato')

# 3. Data Split

In [ ]:
!pip install split-folders

In [ ]:
import splitfolders

splitfolders.ratio(
    IMAGES_FOLDERS_PATH,
    output=OUTPUT_FOLDER,
    seed=42,
    ratio=(0.7, 0.15, 0.15)
)

train_path = os.path.join(OUTPUT_FOLDER, 'train')
val_path   = os.path.join(OUTPUT_FOLDER, 'val')
test_path  = os.path.join(OUTPUT_FOLDER, 'test')

print(f'Train : {train_path}')
print(f'Val   : {val_path}')
print(f'Test  : {test_path}')


# 4. Feature Extraction (PCA)

In [ ]:
from skimage.feature import local_binary_pattern

def compute_color_histogram(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    hist_features = []
    for i in range(3):
        hist = cv2.calcHist([img_rgb], [i], None, [256], [0, 256])
        hist = cv2.normalize(hist, hist).flatten()
        hist_features.extend(hist)
    return hist_features

def compute_texture_features(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, (WIDTH, HEIGHT))

    lbp = local_binary_pattern(gray, LBP_N_POINTS, LBP_RADIUS, LBP_METHOD)
    n_bins = LBP_N_POINTS + 2
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=n_bins,
                                range=(0, n_bins), density=True)

    gray_small = (gray // 4).astype(np.uint8)
    levels = 64
    glcm   = np.zeros((levels, levels), dtype=np.float64)
    glcm[gray_small[:, :-1].ravel(),
         gray_small[:, 1: ].ravel()] += 1
    glcm += glcm.T
    glcm /= glcm.sum()

    i, j  = np.mgrid[0:levels, 0:levels]
    mu    = (i * glcm).sum()
    denom = np.sqrt((glcm * (i - mu)**2).sum()) * \
            np.sqrt((glcm * (j - mu)**2).sum())

    contrast    = (glcm * (i - j)**2).sum()
    energy      = (glcm**2).sum()
    homogeneity = (glcm / (1.0 + (i - j)**2)).sum()
    correlation = (glcm * (i - mu) * (j - mu)).sum() / (denom + 1e-10)

    return list(lbp_hist) + [contrast, energy, homogeneity, correlation]

In [ ]:
all_images = []
for class_name in os.listdir(IMAGES_FOLDERS_PATH):
    class_path = os.path.join(IMAGES_FOLDERS_PATH, class_name)
    if not os.path.isdir(class_path):
        continue
    for img_name in os.listdir(class_path):
        all_images.append((class_name, os.path.join(class_path, img_name)))

print(f"Total images: {len(all_images)}")

In [ ]:
!pip install tqdm

In [ ]:
from tqdm import tqdm

data = []

for class_name, img_path in tqdm(all_images, desc='Extracting features'):
    color_features   = compute_color_histogram(img_path)
    texture_features = compute_texture_features(img_path)

    if color_features is None or texture_features is None:
        continue

    final_features = color_features + texture_features
    data.append({'class': class_name, 'features': final_features})

df = pd.DataFrame(data)
df.to_pickle('features.pkl')

print(f'Number of Images: {len(df)}')
print(df.head())


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from tqdm import tqdm
from skimage.feature import local_binary_pattern

LEVELS = 64
HIST_BINS = 32  # was 256 -> 96 color features instead of 768

def extract_features(class_name, image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None

    # Resize once, up front, so everything below works on a small image
    img = cv2.resize(img, (WIDTH, HEIGHT), interpolation=cv2.INTER_AREA)

    # ---- Color histogram (single pass, no BGR->RGB needed for histograms) ----
    color = []
    for c in range(3):
        h = cv2.calcHist([img], [c], None, [HIST_BINS], [0, 256])
        color.extend(cv2.normalize(h, h).flatten())

    # ---- Texture ----
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    lbp = local_binary_pattern(gray, LBP_N_POINTS, LBP_RADIUS, LBP_METHOD)
    n_bins = LBP_N_POINTS + 2
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=n_bins,
                               range=(0, n_bins), density=True)

    # GLCM via bincount (fast, and correctly counts repeated pairs)
    g = (gray // (256 // LEVELS)).astype(np.int64)
    pairs = g[:, :-1].ravel() * LEVELS + g[:, 1:].ravel()
    glcm = np.bincount(pairs, minlength=LEVELS * LEVELS) \
             .reshape(LEVELS, LEVELS).astype(np.float64)
    glcm += glcm.T
    glcm /= glcm.sum()

    i, j = np.mgrid[0:LEVELS, 0:LEVELS]
    mu = (i * glcm).sum()
    denom = np.sqrt((glcm * (i - mu) ** 2).sum()) * \
            np.sqrt((glcm * (j - mu) ** 2).sum())

    contrast    = (glcm * (i - j) ** 2).sum()
    energy      = (glcm ** 2).sum()
    homogeneity = (glcm / (1.0 + (i - j) ** 2)).sum()
    correlation = (glcm * (i - mu) * (j - mu)).sum() / (denom + 1e-10)

    feats = color + list(lbp_hist) + [contrast, energy, homogeneity, correlation]
    return {'class': class_name, 'features': feats}


# ---- Collect paths ----
all_images = [
    (cls, os.path.join(IMAGES_FOLDERS_PATH, cls, f))
    for cls in os.listdir(IMAGES_FOLDERS_PATH)
    if os.path.isdir(os.path.join(IMAGES_FOLDERS_PATH, cls))
    for f in os.listdir(os.path.join(IMAGES_FOLDERS_PATH, cls))
]
print(f"Total images: {len(all_images)}")

# ---- Run in parallel across all CPU cores ----
results = Parallel(n_jobs=-1, batch_size=32)(
    delayed(extract_features)(c, p) for c, p in tqdm(all_images, desc='Extracting features')
)

df = pd.DataFrame([r for r in results if r is not None])
df.to_pickle('features.pkl')
print(f'Number of Images: {len(df)}')
print(df.head())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

df = pd.read_pickle('features.pkl')
X  = np.stack(df['features'].values)
y  = df['class'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

pca = PCA(n_components=40, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

print(f'Original features : {X_train.shape[1]}')
print(f'After PCA         : {X_train_pca.shape[1]}')


# KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_pca, y_train)
knn_preds = knn.predict(X_test_pca)
print(f"KNN Accuracy: {accuracy_score(y_test, knn_preds) * 100:.2f}%")

import joblib

joblib.dump(knn, "knn_model.joblib")

knn = joblib.load("knn_model.joblib")
print(knn.predict(X_test_pca[:5]))

# First Model

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class ImageFeatureExtractor(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        width=128,
        height=128,
        lbp_points=24,
        lbp_radius=3,
        lbp_method="uniform"
    ):
        self.width = width
        self.height = height
        self.lbp_points = lbp_points
        self.lbp_radius = lbp_radius
        self.lbp_method = lbp_method

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        features = []

        for image_path in X:
            # Read image
            img = cv2.imread(str(image_path))

            if img is None:
                raise ValueError(f"Cannot read image: {image_path}")

            # -------------------------
            # 1. Color histogram
            # -------------------------
            img_rgb = cv2.cvtColor(
                img, cv2.COLOR_BGR2RGB
            )

            color_features = []

            for channel in range(3):
                hist = cv2.calcHist(
                    [img_rgb], [channel],
                    None, [256], [0, 256]
                )

                hist = cv2.normalize(
                    hist, hist
                ).flatten()

                color_features.extend(hist)

            # -------------------------
            # 2. Texture features
            # -------------------------
            gray = cv2.cvtColor(
                img, cv2.COLOR_BGR2GRAY
            )

            gray = cv2.resize(
                gray, (self.width, self.height)
            )

            # Local Binary Pattern
            lbp = local_binary_pattern(
                gray,
                self.lbp_points,
                self.lbp_radius,
                method=self.lbp_method
            )

            n_bins = self.lbp_points + 2

            lbp_hist, _ = np.histogram(
                lbp.ravel(),
                bins=n_bins,
                range=(0, n_bins),
                density=True
            )

            # Gray-Level Co-occurrence Matrix
            gray_small = (gray // 4).astype(np.uint8)
            levels = 64

            glcm = np.zeros(
                (levels, levels), dtype=np.float64
            )

            glcm[
                gray_small[:, :-1].ravel(),
                gray_small[:, 1:].ravel()
            ] += 1

            glcm += glcm.T
            glcm /= glcm.sum()

            i, j = np.mgrid[0:levels, 0:levels]

            mu_i = (i * glcm).sum()
            mu_j = (j * glcm).sum()

            denom = (
                np.sqrt((glcm * (i - mu_i) ** 2).sum())
                * np.sqrt((glcm * (j - mu_j) ** 2).sum())
            )

            contrast = (
                glcm * (i - j) ** 2
            ).sum()

            energy = (glcm ** 2).sum()

            homogeneity = (
                glcm / (1 + (i - j) ** 2)
            ).sum()

            correlation = (
                glcm * (i - mu_i) * (j - mu_j)
            ).sum() / (denom + 1e-10)

            texture_features = (
                list(lbp_hist)
                + [
                    contrast,
                    energy,
                    homogeneity,
                    correlation
                ]
            )

            features.append(
                list(color_features) + texture_features
            )

        return np.asarray(features, dtype=np.float64)

In [ ]:
all_images = []

for class_name in sorted(os.listdir(IMAGES_FOLDERS_PATH)):
    class_path = os.path.join(
        IMAGES_FOLDERS_PATH, class_name
    )

    if not os.path.isdir(class_path):
        continue

    for img_name in sorted(os.listdir(class_path)):
        if img_name.lower().endswith(
            (".jpg", ".jpeg", ".png", ".bmp", ".webp")
        ):
            img_path = os.path.join(
                class_path, img_name
            )

            all_images.append((img_path, class_name))

X = np.array([item[0] for item in all_images])
y = np.array([item[1] for item in all_images])

print("Total images:", len(X))
print("Classes:", np.unique(y))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training images:", len(X_train))
print("Testing images:", len(X_test))

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    (
        "features",
        ImageFeatureExtractor(
            width=WIDTH,
            height=HEIGHT,
            lbp_points=LBP_N_POINTS,
            lbp_radius=LBP_RADIUS,
            lbp_method=LBP_METHOD
        )
    ),
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=40, random_state=42)),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])

In [ ]:
pipeline.fit(X_train, y_train)

print("Pipeline training complete!")

In [ ]:
from sklearn.metrics import classification_report

y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print(f"Test Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred, zero_division=0
))

In [ ]:
import joblib
from sklearn.pipeline import make_pipeline

joblib.dump(pipeline, "knn_pipeline.joblib")

# Load it later
pipeline = joblib.load("knn_pipeline.joblib")

# Predict from the raw (unscaled, un-PCA'd) features
print(pipeline.predict(X_test[:5]))

# CNN Model

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

WIDTH = 224
HEIGHT = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

TRAIN_DATA_TRANSFORMER = transforms.Compose([
    transforms.Resize((HEIGHT, WIDTH)),
    transforms.TrivialAugmentWide(num_magnitude_bins=31),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD)
])

VAL_DATA_TRANSFORMER = transforms.Compose([
    transforms.Resize((HEIGHT, WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD)
])

train_dataset = ImageFolder(
    root=train_path,
    transform=TRAIN_DATA_TRANSFORMER
)

val_dataset = ImageFolder(
    root=val_path,
    transform=VAL_DATA_TRANSFORMER
)

assert train_dataset.class_to_idx == val_dataset.class_to_idx, (
    "Training and validation classes or their index mappings differ."
)

class_names = train_dataset.classes
num_classes = len(class_names)

TRAIN_DATA_LOADER = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)

VAL_DATA_LOADER = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)

print("Classes:", class_names)
print("Number of classes:", num_classes)
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))

In [ ]:
class VGG_CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1: 3 -> 32 channels
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 2: 32 -> 64 channels
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 3: 64 -> 128 channels
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 4: 128 -> 256 channels
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Output becomes 256 x 1 x 1
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = VGG_CNN(num_classes=num_classes).to(device)

# Verify the output shape
sample_images, sample_labels = next(iter(TRAIN_DATA_LOADER))
sample_images = sample_images.to(device)

with torch.no_grad():
    sample_output = model(sample_images)

print("Input shape:", sample_images.shape)
print("Output shape:", sample_output.shape)
print(model)

In [ ]:
loss_fn = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-3
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.1,
    patience=3
)

In [ ]:
def train_step(model, dataloader, loss_fn, optimizer, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for X, y in dataloader:
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(X)
        loss = loss_fn(logits, y)

        loss.backward()
        optimizer.step()

        batch_size = y.size(0)
        total_loss += loss.item() * batch_size
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += batch_size

    return total_loss / total, correct / total


def val_step(model, dataloader, loss_fn, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.inference_mode():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)

            logits = model(X)
            loss = loss_fn(logits, y)

            batch_size = y.size(0)
            total_loss += loss.item() * batch_size
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += batch_size

    return total_loss / total, correct / total

In [ ]:
def train_model(
    model,
    train_loader,
    val_loader,
    loss_fn,
    optimizer,
    scheduler,
    epochs=30,
    patience=5,
    save_path="best_cnn_model.pth"
):
    results = {
        "Train Loss": [],
        "Train Accuracy": [],
        "Val Loss": [],
        "Val Accuracy": []
    }

    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    patience_counter = 0

    for epoch in range(epochs):
        train_loss, train_acc = train_step(
            model, train_loader, loss_fn, optimizer, device
        )

        val_loss, val_acc = val_step(
            model, val_loader, loss_fn, device
        )

        scheduler.step(val_loss)

        results["Train Loss"].append(train_loss)
        results["Train Accuracy"].append(train_acc)
        results["Val Loss"].append(val_loss)
        results["Val Accuracy"].append(val_acc)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, save_path)
            patience_counter = 0
        else:
            patience_counter += 1

        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.2%} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.2%} | "
            f"LR: {current_lr:.2e}"
        )

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch + 1}")
            break

    model.load_state_dict(best_state)
    return results

In [ ]:
SAVE_PATH = "/content/best_cnn_model.pth"

results = train_model(
    model=model,
    train_loader=TRAIN_DATA_LOADER,
    val_loader=VAL_DATA_LOADER,
    loss_fn=loss_fn,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=30,
    patience=5,
    save_path=SAVE_PATH
)

In [ ]:
results_df = pd.DataFrame(results)
display(results_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(results["Train Loss"], label="Train")
axes[0].plot(results["Val Loss"], label="Validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(results["Train Accuracy"], label="Train")
axes[1].plot(results["Val Accuracy"], label="Validation")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

best_epoch = int(np.argmin(results["Val Loss"]))
best_val_accuracy = results["Val Accuracy"][best_epoch]

print("Best epoch:", best_epoch + 1)
print(f"Validation accuracy at best epoch: {best_val_accuracy:.2%}")

# CNN tensorflow

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

SEED = 42

tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

# Update these paths to match your dataset
TRAIN_DATA_PATH = train_path
VAL_DATA_PATH = val_path

WIDTH = 224
HEIGHT = 224
BATCH_SIZE = 32
EPOCHS = 30

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DATA_PATH,
    image_size=(HEIGHT, WIDTH),
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True,
    seed=SEED
)

class_names = train_ds.class_names
num_classes = len(class_names)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DATA_PATH,
    image_size=(HEIGHT, WIDTH),
    batch_size=BATCH_SIZE,
    label_mode="int",
    class_names=class_names,
    shuffle=False
)

print("Classes:", class_names)
print("Number of classes:", num_classes)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
], name="data_augmentation")


model = keras.Sequential([
    layers.Input(shape=(HEIGHT, WIDTH, 3)),

    # Applied only during training
    data_augmentation,

    # Normalize pixel values from 0–255 to 0–1
    layers.Rescaling(1.0 / 255),

    # Convolution block 1
    layers.Conv2D(32, 3, padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D(2),

    # Convolution block 2
    layers.Conv2D(64, 3, padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D(2),

    # Convolution block 3
    layers.Conv2D(128, 3, padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D(2),

    # Convolution block 4
    layers.Conv2D(256, 3, padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D(2),

    # Classification head
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation="softmax")
], name="custom_cnn")

model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

In [ ]:
CHECKPOINT_PATH = "/content/best_cnn_model.keras"

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=CHECKPOINT_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.1,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

In [ ]:
import tensorflow as tf

print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

In [ ]:
history_df = pd.DataFrame(history.history)

display(history_df.tail())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history.history["accuracy"], label="Training")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set_title("CNN Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"], label="Training")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set_title("CNN Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

best_epoch = np.argmin(history.history["val_loss"]) + 1
best_val_accuracy = history.history["val_accuracy"][best_epoch - 1]

print("Best epoch:", best_epoch)
print(f"Validation accuracy at best-loss epoch: {best_val_accuracy:.2%}")

In [ ]:
y_true = np.concatenate([
    labels.numpy()
    for _, labels in val_ds
])

y_prob = model.predict(val_ds, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

print("\nClassification Report:\n")

print(classification_report(
    y_true,
    y_pred,
    labels=np.arange(num_classes),
    target_names=class_names,
    zero_division=0
))

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=np.arange(num_classes)
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

fig, ax = plt.subplots(figsize=(9, 9))
disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title("CNN Validation Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
MODEL_PATH = "/content/cnn_model.keras"
CLASS_NAMES_PATH = "/content/cnn_class_names.json"

model.save(MODEL_PATH)

with open(CLASS_NAMES_PATH, "w") as f:
    json.dump(class_names, f)

print("Saved model:", MODEL_PATH)
print("Saved class names:", CLASS_NAMES_PATH)

In [ ]:
def predict_image(image_path, model, class_names):
    image = tf.keras.utils.load_img(
        image_path,
        target_size=(HEIGHT, WIDTH),
        color_mode="rgb"
    )

    image_array = tf.keras.utils.img_to_array(image)
    image_array = np.expand_dims(image_array, axis=0)

    probabilities = model.predict(image_array, verbose=0)[0]
    predicted_index = int(np.argmax(probabilities))

    predicted_class = class_names[predicted_index]
    confidence = float(probabilities[predicted_index])

    print("Predicted class:", predicted_class)
    print(f"Confidence: {confidence:.2%}")

    plt.figure(figsize=(5, 5))
    plt.imshow(image)
    plt.title(f"{predicted_class} ({confidence:.1%})")
    plt.axis("off")
    plt.show()

    return predicted_class, probabilities


# Example:
# predict_image(
#     "/content/drive/MyDrive/test_image.jpg",
#     model,
#     class_names
# )

In [ ]:
predict_image(
    "/content/plantdisease_data/PlantVillage/Pepper__bell___Bacterial_spot/0022d6b7-d47c-4ee2-ae9a-392a53f48647___JR_B.Spot 8964.JPG",
    model,
    class_names
)